# PI follow-up analyses — sameloc model

Five analyses the PI requested, on the sameloc go/no-go model
(`CuedTargetSameLoc2D`, `stage1_topo_sameloc.pt`):

1. Per-neuron position tuning (offset vs position×time interaction).
2. Position dPCA — split variance into position `s`, time `t`, interaction `ts`.
3. Participation ratio vs **time from cue** (real buildup or noise?).
4. Difference maps between timepoints (`t=+100 ms` − earlier `t`) → localised bump.
5. Normalization check.

Everything is driven by one `CKPT` path, so re-running on a future COM-loss
checkpoint only needs that line changed.

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetSameLoc2D
from src.analysis import (
    collect_trials_spatial, filter_trials, participation_ratio,
    collect_aligned_hidden_by_label, make_condition_mean_tensor,
)

device = "cpu"
DT = 20
CKPT = "checkpoints/stage1_topo_sameloc.pt"  # repoint here for a COM-loss model


def make_model():
    return BioLeakyRNNTopo(
        input_size=7, hidden_size=180, output_size=2, dt=20.0, tau=100.0,
        activation="softplus", sigma_rec=0.10, rec_init="diag", use_ei=True,
        exc_ratio=0.80, use_dale=True, mask_seed=42, sheet_side=12,
        tau_ee=0.25, tau_ie=0.32, tau_ei=0.64, tau_ii=0.64, rf_sigma=0.3,
        tau_e_range=(50, 150), tau_i_range=(10, 50), use_adaptation=False,
    ).to(device)


model = make_model()
ckpt = torch.load(CKPT, weights_only=True)
model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval()
model.noise_at_eval = True

H = int(model.hidden_size)
n_exc = int(model.n_exc)
coords = model.coords.detach().cpu().numpy()


def make_env():
    return CuedTargetSameLoc2D(dt=20, cue_strength=1.0,
                               p_distractor_trial=0.0, distractor_strength=0.0)


torch.manual_seed(0); np.random.seed(0)
trials = collect_trials_spatial(model, make_env, n_trials=2500, device=device, head="go_nogo")
corr = filter_trials(trials, outcome="correct")
print(f"correct: {len(corr)} / {len(trials)}")

N_POS = 4  # position bins (quadrants by angle)


def theta_bin(pos, n=N_POS):
    x, y = pos
    th = np.arctan2(y, x)
    return int((th + np.pi) / (2 * np.pi) * n) % n


pos_counts = defaultdict(int)
for tr in corr:
    pos_counts[theta_bin(tr["target_pos"])] += 1
print("position-bin counts:", dict(sorted(pos_counts.items())))

bin_counts = defaultdict(int)
for tr in corr:
    bin_counts[tr["ctoa_bin"]] += 1
rep_bin = max(bin_counts, key=bin_counts.get)
print("CTOA bin counts:", dict(sorted(bin_counts.items())), "| rep_bin:", rep_bin)

## 1. Per-neuron position tuning

For the best position-tuned neurons, mean activity aligned to **cue onset**, one curve
per position bin (post-target timepoints masked out so we see delay buildup only).
**Parallel curves → pure position offset; different slopes → position×time interaction.**

In [ ]:
def neuron_eta(trials, n_exc, n_bins=N_POS, w=10):
    """eta^2 location selectivity per E neuron on the post-target window."""
    by = defaultdict(list)
    for tr in trials:
        by[theta_bin(tr["target_pos"], n_bins)].append(tr)
    bns = [b for b in sorted(by) if len(by[b]) >= 10]
    eta = np.zeros(n_exc)
    for i in range(n_exc):
        groups = [np.array([tr["h"][tr["target_onset"]:tr["target_onset"] + w, i].mean()
                            for tr in by[b]]) for b in bns]
        allv = np.concatenate(groups); gm = allv.mean()
        ssb = sum(len(g) * (g.mean() - gm) ** 2 for g in groups)
        sst = ((allv - gm) ** 2).sum()
        eta[i] = ssb / sst if sst > 0 else 0.0
    return eta


eta = neuron_eta(corr, n_exc)
top = np.argsort(eta)[::-1][:3]
print("top-3 tuned E neurons:", top, "eta:", np.round(eta[top], 3))

# Cue-aligned curves, delay only (mask t >= target_onset).
CB, CA = 10, 80  # -200 .. +1600 ms from cue
rel_ms = np.arange(-CB, CA + 1) * DT
by_pos = defaultdict(list)
for tr in corr:
    by_pos[theta_bin(tr["target_pos"])].append(tr)

colors = plt.cm.viridis(np.linspace(0.1, 0.9, N_POS))
fig, axes = plt.subplots(1, len(top), figsize=(5 * len(top), 4), sharex=True)
for ax, nrn in zip(np.atleast_1d(axes), top):
    for b in range(N_POS):
        segs = []
        for tr in by_pos[b]:
            c0 = tr["cue_onset"]; t0 = tr["target_onset"]; h = tr["h"]
            s, e = c0 - CB, c0 + CA + 1
            if s < 0 or e > len(h):
                continue
            seg = h[s:e, nrn].astype(float).copy()
            seg[(np.arange(s, e) >= t0)] = np.nan   # delay only
            segs.append(seg)
        if segs:
            m = np.nanmean(np.stack(segs), axis=0)
            ax.plot(rel_ms, m, color=colors[b], lw=2, label=f"pos bin {b}")
    ax.axvline(0, ls="--", c="grey", lw=0.8)
    ax.set_xlabel("time from cue (ms)"); ax.set_title(f"neuron {nrn} (eta^2={eta[nrn]:.2f})")
    ax.legend(fontsize=8)
axes if isinstance(axes, np.ndarray) else None
np.atleast_1d(axes)[0].set_ylabel("activity")
plt.tight_layout(); plt.show()

## 2. Position dPCA (s / t / ts)

Condition = position bin. Orthogonal SS split of the target-locked window into
condition-independent time `t`, pure position offset `s`, and **position×time
interaction `ts`** — the headline number the PI asked for.

In [ ]:
from matplotlib.patches import Patch

by, _ = collect_aligned_hidden_by_label(
    corr, label_fn=lambda tr: theta_bin(tr["target_pos"]),
    align_key="target_onset", window_before=20, window_after=20)
X, labels, counts = make_condition_mean_tensor(by, min_trials=5)  # [C, T, H]
print("position conditions:", labels, "counts:", counts)

mu = X.mean((0, 1), keepdims=True)
phi_t = X.mean(0, keepdims=True) - mu        # condition-independent time
phi_p = X.mean(1, keepdims=True) - mu        # pure position offset s
phi_pt = X - mu - phi_t - phi_p              # position x time interaction ts
ss = lambda a: float((np.broadcast_to(a, X.shape) ** 2).sum())
t, s, ts = ss(phi_t), ss(phi_p), ss(phi_pt)
tot = t + s + ts
print(f"POSITION dPCA  ->  time t={t/tot:.1%}   pure-position s={s/tot:.1%}   interaction ts={ts/tot:.1%}")
print(f"   position-related (s+ts) = {(s+ts)/tot:.1%}   |  interaction ts alone = {ts/tot:.1%}")

GREY, GOLD, ORANGE = "#bdbdbd", "#f0c05a", "#e8845b"
pct = dict(autopct="%1.0f%%", pctdistance=0.8, startangle=90, counterclock=False,
           wedgeprops=dict(width=0.45, edgecolor="w"),
           textprops=dict(color="white", fontsize=11, fontweight="bold"))
fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
ax[0].pie([t / tot * 100, (s + ts) / tot * 100], colors=[GREY, ORANGE], **pct)
ax[0].set_title("time vs position-related")
ax[1].pie([t / tot * 100, s / tot * 100, ts / tot * 100], colors=[GREY, GOLD, ORANGE], **pct)
ax[1].set_title("t / s / ts")
fig.legend(handles=[Patch(color=GREY, label="condition-independent time (t)"),
                    Patch(color=GOLD, label="pure-position offset (s)"),
                    Patch(color=ORANGE, label="position x time interaction (ts)")],
           loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Position dPCA variance split (sameloc)", fontsize=13)
fig.subplots_adjust(bottom=0.16, top=0.86)
plt.show()

## 3. Participation ratio vs time from cue

PR computed across trials at each timepoint, aligned to cue onset, separately for
early / mid / late CTOA groups. Shows whether dimensionality genuinely **builds up**
through the delay (PI's question), not as one per-bin point.

In [ ]:
def boot_pr_ci(Xt, n_boot=50, seed=0):
    rng = np.random.default_rng(seed)
    n = Xt.shape[0]
    vals = [participation_ratio(Xt[rng.integers(0, n, n)]) for _ in range(n_boot)]
    return np.percentile(vals, [2.5, 97.5])


groups = {"early CTOA (0-2)": range(0, 3), "mid (4-6)": range(4, 7), "late (7-9)": range(7, 10)}
gcolors = {"early CTOA (0-2)": "#5b9bd5", "mid (4-6)": "#70ad47", "late (7-9)": "#ed7d31"}

CB, CA = 10, 80
rel_ms = np.arange(-CB, CA + 1) * DT
CI_STRIDE = 4   # bootstrap CI only every Nth timepoint (cheap); PR curve stays full-res
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for gname, brange in groups.items():
    bset = set(brange)
    segs = []
    for tr in corr:
        if tr["ctoa_bin"] not in bset:
            continue
        c0 = tr["cue_onset"]; t0 = tr["target_onset"]; h = tr["h"]
        s, e = c0 - CB, c0 + CA + 1
        if s < 0 or e > len(h):
            continue
        seg = h[s:e].astype(float).copy()
        seg[np.arange(s, e) >= t0] = np.nan   # delay only
        segs.append(seg)
    if len(segs) < 10:
        continue
    S = np.stack(segs)                        # [trials, T, H]
    pr = np.full(S.shape[1], np.nan)
    ci_t, ci_lo, ci_hi = [], [], []
    for ti in range(S.shape[1]):
        Xt = S[:, ti, :]
        Xt = Xt[~np.isnan(Xt).any(axis=1)]    # trials still in delay at this time
        if Xt.shape[0] < 10:
            continue
        pr[ti] = participation_ratio(Xt)
        if ti % CI_STRIDE == 0:               # bootstrap CI on a coarse grid only
            lo, hi = boot_pr_ci(Xt)
            ci_t.append(rel_ms[ti]); ci_lo.append(lo); ci_hi.append(hi)
    ax.plot(rel_ms, pr, color=gcolors[gname], lw=2, label=f"{gname} (n={len(segs)})")
    if ci_t:
        ax.fill_between(ci_t, ci_lo, ci_hi, color=gcolors[gname], alpha=0.2)
ax.axvline(0, ls="--", c="grey", lw=0.8)
ax.set_xlabel("time from cue (ms)"); ax.set_ylabel("participation ratio")
ax.set_title("PR vs time from cue (delay only) — buildup?")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 3b. Control — is the buildup dynamics or position-spread?

Pooled-position PR mixes two sources: genuine within-trial temporal buildup, and the
fact that trials at different positions diverge over the delay. Holding the CTOA group
fixed (mid, bins 4–6), compare **pooled-position** PR(t) against the **mean
within-position** PR(t). If within-position is much lower/flatter, the buildup is mostly
position-spread, not dynamics.

In [ ]:
    CB, CA = 10, 80
rel_ms = np.arange(-CB, CA + 1) * DT
GROUP = {4, 5, 6}   # fixed CTOA group, so only position-pooling differs


def _seg_delay(tr):
    c0 = tr["cue_onset"]; t0 = tr["target_onset"]; h = tr["h"]
    s, e = c0 - CB, c0 + CA + 1
    if s < 0 or e > len(h):
        return None
    seg = h[s:e].astype(float).copy()
    seg[np.arange(s, e) >= t0] = np.nan   # delay only
    return seg


def _pr_curve(trial_list):
    segs = [s for s in (_seg_delay(tr) for tr in trial_list) if s is not None]
    if len(segs) < 10:
        return np.full(CB + CA + 1, np.nan)
    S = np.stack(segs)
    out = []
    for ti in range(S.shape[1]):
        Xt = S[:, ti, :]; Xt = Xt[~np.isnan(Xt).any(axis=1)]
        out.append(participation_ratio(Xt) if Xt.shape[0] >= 10 else np.nan)
    return np.array(out)


mid = [tr for tr in corr if tr["ctoa_bin"] in GROUP]
pooled = _pr_curve(mid)                                   # positions mixed in
within = [_pr_curve([tr for tr in mid if theta_bin(tr["target_pos"]) == b])
          for b in range(N_POS)]
within_mean = np.nanmean(np.stack(within), axis=0)        # position spread removed

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(rel_ms, pooled, color="#d62728", lw=2, label="across positions (pooled)")
ax.plot(rel_ms, within_mean, color="#1f77b4", lw=2, label="within position (mean over bins)")
ax.axvline(0, ls="--", c="grey", lw=0.8)
ax.set_xlabel("time from cue (ms)"); ax.set_ylabel("participation ratio")
ax.set_title("PR buildup: dynamics vs position-spread (mid CTOA)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"end-of-delay PR  pooled={np.nanmax(pooled):.2f}   within-position={np.nanmax(within_mean):.2f}")
print("within << pooled  => buildup is position-spread;  within ~ pooled  => genuine dynamics")

## 4. Difference maps (t=+100 ms − earlier t)

Per-neuron pre-cue baseline removed, then `map(+100 ms) − map(t_early)` on the sheet
to expose a localised spatial bump. Expected to be clearer on a COM-loss model — just
repoint `CKPT` and re-run.

In [ ]:
def smooth_map(vals, pts, grid_size=80, sigma=0.12):
    """Toroidal Gaussian smoothing of per-neuron values onto a grid."""
    g = np.linspace(-1, 1, grid_size)
    GX, GY = np.meshgrid(g, g)
    Z = np.zeros_like(GX)
    for i in range(len(vals)):
        dx = GX - pts[i, 0]; dx -= 2.0 * np.round(dx / 2.0)
        dy = GY - pts[i, 1]; dy -= 2.0 * np.round(dy / 2.0)
        Z += vals[i] * np.exp(-(dx ** 2 + dy ** 2) / (2 * sigma ** 2))
    return Z


T_EARLY_MS, T_LATE_MS = -200, 100   # relative to target onset
rep_trials = [tr for tr in corr if tr["ctoa_bin"] == rep_bin]
by_pos_rep = defaultdict(list)
for tr in rep_trials:
    by_pos_rep[theta_bin(tr["target_pos"])].append(tr)

fig, axes = plt.subplots(1, N_POS, figsize=(3.2 * N_POS, 3.4))
M = 0.0
panels = {}
for b in range(N_POS):
    segs = []
    for tr in by_pos_rep[b]:
        t0 = tr["target_onset"]; h = tr["h"]
        ie, il = t0 + T_EARLY_MS // DT, t0 + T_LATE_MS // DT
        if ie < tr["cue_onset"] or il >= len(h):
            continue
        base = h[:tr["cue_onset"]].mean(axis=0)
        segs.append((h[il] - base) - (h[ie] - base))   # == h[il] - h[ie]
    if not segs:
        panels[b] = None; continue
    d = np.mean(segs, axis=0)
    Z = smooth_map(d, coords)
    panels[b] = Z
    M = max(M, np.percentile(np.abs(Z), 99))
M = M or 1.0
for b in range(N_POS):
    ax = axes[b]
    if panels[b] is None:
        ax.set_visible(False); continue
    ax.imshow(panels[b], origin="lower", extent=[-1, 1, -1, 1], cmap="RdBu_r", vmin=-M, vmax=M)
    tx, ty = np.mean([tr["target_pos"] for tr in by_pos_rep[b]], axis=0)
    ax.plot(tx, ty, "o", mfc="none", mec="k", mew=1.5, ms=12)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"pos bin {b}")
fig.suptitle(f"Difference maps  (+{T_LATE_MS} ms) - ({T_EARLY_MS} ms) | CTOA bin {rep_bin}", fontsize=12)
plt.tight_layout()
fig.savefig("figures/14_diff_maps.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Normalization check

(a) total population activity over time (implicit normalization?); (b) per-neuron
baseline vs modulation depth — are high-baseline "stable" units normalization
artifacts?; (c) state norm stays bounded (no explosive growth).

In [ ]:
CB, CA = 10, 80
rel_ms = np.arange(-CB, CA + 1) * DT
relu = lambda x: np.clip(x, 0, None)

tot_act, hnorm = [], []
base_fr, mod_depth = np.zeros(H), np.zeros(H)
nb = np.zeros(H); nm = np.zeros(H)
for tr in corr:
    c0 = tr["cue_onset"]; t0 = tr["target_onset"]; h = tr["h"]
    s, e = c0 - CB, c0 + CA + 1
    if s >= 0 and e <= len(h):
        seg = h[s:e].astype(float).copy()
        mask = np.arange(s, e) >= t0
        seg[mask] = np.nan
        tot_act.append(relu(seg).sum(axis=1))
        hnorm.append(np.sqrt((seg ** 2).sum(axis=1)))
    base = h[:c0].mean(axis=0)
    dly = h[c0:t0]
    if len(dly) > 1:
        base_fr += base; nb += 1
        mod_depth += dly.max(axis=0) - dly.min(axis=0); nm += 1
base_fr /= np.maximum(nb, 1); mod_depth /= np.maximum(nm, 1)
tot_act = np.nanmean(np.stack(tot_act), axis=0)
hnorm = np.nanmean(np.stack(hnorm), axis=0)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(rel_ms, tot_act, color="#1f77b4"); ax[0].axvline(0, ls="--", c="grey")
ax[0].set_xlabel("time from cue (ms)"); ax[0].set_ylabel("sum relu(h)")
ax[0].set_title("Total population activity (conserved?)"); ax[0].grid(alpha=0.3)

ax[1].scatter(base_fr[:n_exc], mod_depth[:n_exc], s=12, c="#ed7d31", label="E")
ax[1].scatter(base_fr[n_exc:], mod_depth[n_exc:], s=12, c="#5b9bd5", marker="^", label="I")
ax[1].set_xlabel("pre-cue baseline activity"); ax[1].set_ylabel("delay modulation depth")
ax[1].set_title("Baseline vs modulation (stable units = low y)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

ax[2].plot(rel_ms, hnorm, color="#2ca02c"); ax[2].axvline(0, ls="--", c="grey")
ax[2].set_xlabel("time from cue (ms)"); ax[2].set_ylabel("||h(t)||")
ax[2].set_title("State norm (bounded? no explosion)"); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

n_stable = int((mod_depth[:n_exc] < 0.05 * (mod_depth[:n_exc].max() + 1e-9)).sum())
print(f"~{n_stable}/{n_exc} E units are near-static in the delay (low modulation depth)")
print(f"total activity range over delay: {np.nanmin(tot_act):.2f} - {np.nanmax(tot_act):.2f}")

## Interpretation (fill after running)

- **Tuning (1) + dPCA (2):** if curves are parallel and `ts` is small (~5% as before),
  position is a static amplitude offset with no position×time interaction — the model
  does not compute differently for different positions over time.
- **PR buildup (3):** does PR rise through the delay across CTOA groups (real buildup) or
  stay flat/noisy?
- **Difference maps (4):** is there a localised bump at the target position? Expected
  weak here, clearer under COM loss.
- **Normalization (5):** is total activity roughly conserved (implicit normalization)?
  Are the "stable" low-modulation units high-baseline (normalization artifact) or just
  unused?